# 6장. 에이전트 호출 순차적으로 연결하기

## 터미널에서 2개의 ACP 서버 실행

```
uv run aidagent_server.py
uv run guideagent_server.py
```

## LLM 호출 연결

In [1]:
import nest_asyncio
nest_asyncio.apply()

In [4]:
from acp_sdk.client import Client
import asyncio
from colorama import Fore

async def run_workflow() -> None:
    async with Client(base_url="http://localhost:8000") as first_aid, Client(base_url="http://localhost:8001") as guide_search:
        run1 = await first_aid.run_sync(
            agent="policy_agent", input="학교 내에서 응급환자 발생 시 응급처치는 어떻게 할 것인가?"
        )
        content = run1.output[0].parts[0].content
        print(Fore.LIGHTMAGENTA_EX+ content + Fore.RESET)

        run2 = await guide_search.run_sync(
            agent="guide_agent", input=f"맥락: {content} 증상별 응급처치에서 우선 처치 사항은 무엇인가?"
        )
        print(Fore.YELLOW + run2.output[0].parts[0].content + Fore.RESET)

In [5]:
asyncio.run(run_workflow())

학교 내에서 응급환자 발생 시 응급처치는 다음과 같은 절차를 따릅니다:

1. 기본 방침:
   - 모든 교내 안전사고 및 응급환자 발생 시에는 교감과 교장에게 신고하고 필요한 지시를 따릅니다.
   - 환자의 활력증상을 측정하고 사정한 후, 상태에 따라 응급처치를 시행합니다.
   - 환자의 활력증상 및 응급처치를 기록합니다.
   - 위급한 경우에는 전문의료기관으로 후송합니다.

2. 위급하거나 위독한 경우:
   - 보건교사는 응급처치 후 119에 연락하거나 다른 후송 방법을 모색합니다.
   - 환자 발생과 후송 방법을 보고합니다.
   - 담임교사는 부모에게 연락합니다. (의료보험카드를 지참하도록 안내합니다)
   - 보건교사와 담임교사가 병원으로 환자를 후송합니다.
   - 학생 진료상황을 학교에 보고하고 학부모를 위로합니다.

3. 위급하지 않으나 병원으로 후송할 경우:
   - 보건교사는 응급처치 후 담임교사에게 알립니다.
   - 환자 발생과 후송 방법을 보고합니다.
   - 담임교사는 부모에게 연락합니다. (의료보험카드를 지참하도록 안내합니다)
   - 보건교사는 부모가 이용할 병원에 알려 신속하게 치료 받을 수 있도록 도움을 줍니다 (단, 병원 선택은 학부모 의견에 따른다).
   - 담임교사 및 학부모와 함께 병원에 갑니다.
   - 보건교사는 다른 응급환자를 위해 교내에 대기합니다.

4. 응급처치 시의 일반적 원칙:
   - 주변환경의 위험성을 파악하고, 가능하면 환자를 안전한 곳으로 옮깁니다.
   - 심호흡정지, 심한 출혈, 쇼크 환자 등 긴급을 요하는 환자부터 우선 처리합니다.
   - 필요시 119에 연락하고, 환자에게 말을 걸어 안정시킵니다.
   - 환자는 편안히 눕히고 보온을 유지합니다.
   - 의식이 없는 환자에게는 아무 것도 주지 않습니다.
   - 응급환자를 지속적으로 관찰하고 기록합니다.

이러한 절차를 통해 학교 내에서 응급환자가 발생했을 때 신속하고 적절한 처치를 통해 생명을 구하고, 손상의 악화를 방지할 수 